# **PointPillar for 3D object detection**

PointPillar is a fast and efficient deep-learning architecture for 3D object detection from LiDAR point clouds, commonly used in autonomous driving.

Instead of operating directly on raw points or dense 3D voxels, PointPillar groups points into vertical columns ("pillars") and encodes per-pillar features. 
These pillar features are arranged into a pseudo-image that a 2D convolutional backbone can process. The pipeline is lightweight and well-suited for real-time inference.

Core stages:
- Voxelization / Pillarization: group points into pillars and compute per-pillar statistics.
- Pillar feature encoding: a small network encodes points in each pillar into a fixed-size feature vector.
- Scatter to pseudo-image: place each pillar's feature into a 2D grid (pseudo-image) based on the pillar's X-Y location.
- 2D backbone + neck: apply 2D convolutions to produce multi-scale feature maps.
- Detection head: predict class scores, bounding box regressions, and directions on the pseudo-image.
- Post-processing: decode boxes, apply non-maximum suppression (NMS), and output final detections.

In this tutorial we consider how to run PointPillars with OpenVINO.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/3D-point-pillars/pointpillars.ipynb" />


#### Table of contents:
1. [Prerequisites](#prerequisites)
2. [Install python packages](#install-python-packages)
3. [Build Extensions](#build-extensions)
4. [Exporting the model](#exporting-the-model)
5. [Inference with OpenVINO](#inference-with-openvino)
6. Utilities
    * [KITTI bin to PCD](#kitti-bin-to-pcd)
### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

#### **Prerequisites:**

```bash
# Install the required packages specific to this notebook.
sudo apt update && sudo apt install -y \
  software-properties-common \
  build-essential \
  cmake \
  git \
  libx11-6 \
  libgl1

# Install Intel GPU runtime for OpenCL for using Intel GPU device with OpenVINO
sudo add-apt-repository -y ppa:kobuk-team/intel-graphics
sudo apt update
sudo apt install -y --no-install-recommends \
    libze-intel-gpu1 \
    intel-opencl-icd

# Create a python 3.10 environment, conda can be used to manage environments:
conda install python=3.10
conda create -n ovpp310 python=3.10
conda activate ovpp310
# conda deactivate
# conda env remove -n ovpp310
```

**Python 3.10** is recommended to run this notebook. We also recommend running the notebook in a _virtual environment_. 
You only need a Jupyter server to start and select python environment in the kernel. 
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).


In [ ]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("pointpillars.ipynb")

In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install

repo_dir = Path("openvino_contrib").resolve(strict=False)
pp_dir = Path(repo_dir, "modules", "3d", "pointPillars")
revision = "962f5e1"
clone_repo("https://github.com/openvinotoolkit/openvino_contrib.git", revision)

[back to top ⬆️](#Table-of-contents:)

#### **Install python packages:**

In [ ]:
# Install the required pip packages specific to this notebook
pip_install("-r", str(pp_dir / "requirements.txt"), "--extra-index-url", "https://download.pytorch.org/whl/cpu")

[back to top ⬆️](#Table-of-contents:)

#### **Build Extensions:**

In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None):
    print("Running:", " ".join(cmd), "in", cwd)
    try:
        subprocess.run(cmd, cwd=cwd, check=True, env=env)
    except subprocess.CalledProcessError as e:
        print("Command failed:", e)
        raise


# Build the openvino extension
run(["rm", "-rf", str(Path(pp_dir, "ov_extensions", "build"))])
run(["bash", "build.sh"], cwd=str(Path(pp_dir, "ov_extensions")))

# Build the pytorch extensions (will be used only to export the model)
run(["rm", "-rf", str(Path(pp_dir, "build"))])
run(["rm", "-rf", str(Path(pp_dir, "pointpillars", "ops", "*.so"))])
env = os.environ.copy()
env["CPU_BUILD"] = "1"
run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=str(pp_dir), env=env)

[back to top ⬆️](#Table-of-contents:)
<div>
<style scoped>
pre[class*="language-python"], code[class*="language-python"] {
  font-size:12px !important;
}
</style>
</div>

#### **Exporting the model:**

In the model declaration, we can see it has five layers. 
Out of these five layers, the first layer is the Pillar Layer where a pytorch extension (voxelization) is used. 
For exporting the model to OpenVINO format, we skip this layer and export the remaining layers to OpenVINO format. 
During inference, we run the Pillar Layer separately using the openvino extension and then pass the output to the OpenVINO model for further processing.

```python
class NeuralNetworkPortion(nn.Module):
  """Neural network portion: PillarEncoder + Backbone + Neck + Head"""

  def __init__(self, model):
    super().__init__()
    self.pillar_encoder = model.pillar_encoder
    self.backbone = model.backbone
    self.neck = model.neck
    self.head = model.head

  def forward(self, pillars, coors, npoints):
    pillar_features = self.pillar_encoder(pillars, coors, npoints)
    xs = self.backbone(pillar_features)
    x = self.neck(xs)
    cls_preds, box_preds, dir_cls_preds = self.head(x)
    return cls_preds, box_preds, dir_cls_preds
```

Getting the neural network portion of the PointPillars model:

```python
full_model = PointPillars()
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
full_model.load_state_dict(checkpoint)
full_model.eval()

nn_portion = NeuralNetworkPortion(full_model)
```

Then defining dummy inputs for the model to export:

```python
dummy_pillars = torch.randn(max_voxels, max_points, 4)
dummy_npoints = torch.randint(1, max_points, (max_voxels,)).long()

vx, vy, vz = voxel_size[0], voxel_size[1], voxel_size[2]
x_l = int((point_cloud_range[3] - point_cloud_range[0]) / vx)
y_l = int((point_cloud_range[4] - point_cloud_range[1]) / vy)
z_l = int((point_cloud_range[5] - point_cloud_range[2]) / vz)
dummy_coors = torch.empty((max_voxels, 4), dtype=torch.long)
dummy_coors[:, 0] = 0  # batch index
if z_l > 0:
    dummy_coors[:, 1] = torch.randint(0, z_l, (max_voxels,))
else:
    dummy_coors[:, 1] = 0
dummy_coors[:, 2] = torch.randint(0, y_l, (max_voxels,))
dummy_coors[:, 3] = torch.randint(0, x_l, (max_voxels,))
```

Finally, exporting the model to OpenVINO format:

```python
with torch.no_grad():
  traced_nn = torch.jit.trace(
      nn_portion,
      (dummy_pillars, dummy_coors, dummy_npoints),
      check_trace=False,
      strict=False
  )
  ov_nn_model = ov.convert_model(
      traced_nn,
      example_input=(dummy_pillars, dummy_coors, dummy_npoints),
      input=[
          ov.PartialShape([-1, max_points, 4]),  # pillars
          ov.PartialShape([-1, 4]),              # coors
          ov.PartialShape([-1]),                 # npoints
      ]
  )
```

And saving the neural network model:

```python
ov.save_model(ov_nn_model, nn_xml_path)
```

The other preprocessing and post-processing steps (including voxelization, non-maximum suppression, etc.) are handled 
separately using OpenVINO custom extensions. The details of the xml and other parameters are implemented in `create_pillar_layer_ir` and `create_postprocessing_ir` functions available in `export_ov_e2e.py`. 


In [ ]:
# Export the PointPillars model to OpenVINO format
run(
    [
        sys.executable,
        "export_ov_e2e.py",
        "--checkpoint",
        str(Path(pp_dir, "pretrained", "epoch_160.pth")),
        "--output",
        str(Path(pp_dir, "pretrained", "pointpillars_ov")),
    ],
    cwd=str(pp_dir),
)

[back to top ⬆️](#Table-of-contents:)
#### **Select inference device**

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])

device.value

[back to top ⬆️](#Table-of-contents:)

#### **Inference with OpenVINO:**

For inference, implemented in `e2eOVInference.py`, first voxelizing the raw point cloud with a custom OpenVINO voxelization op, then running the compiled neural network IR on the resulting pillars/coors/npoints, and finally applying a custom OpenVINO post-processing op to decode boxes, apply NMS and output final bboxes, labels and scores.

**Setup (`__init__`):**

Load the JSON config produced by export_ov_e2e.py (paths to IRs and extension library) and create an OpenVINO Core instance.
Load the custom extension library and compile three models: voxelization (CPU), neural network (device, e.g. CPU/GPU), and post-processing (CPU).

**Step 1 — Voxelization (custom OpenVINO op):**

Call compiled voxel model with the raw points, a filtered point cloud as an N×4 float array (x, y, z, intensity).
Output: pillars (float tensor, each row holds the point for one pillar, zero‑padded to max_points), coors (int tensor, grid coordinates for each pillar), npoints (int tensor, number of valid points in each corresponding pillar).

**Step 2 — Neural network inference:**

Feed pillars, coors, npoints into the compiled NN IR.
Output: `cls_preds`, `box_preds`, `dir_cls_preds` (network predictions).

**Step 3 — Post-processing (custom OpenVINO op):**

Call compiled `postproc_model` with the network outputs.
Output: final bboxes, labels, and scores (already decoded, NMS applied, rotated IoU handled by the custom op).
Post-processing parameters (anchors, NMS thresholds, etc.) come from the config exported earlier.

In [ ]:
# Run inference with OpenVINO
run(
    [
        sys.executable,
        "e2eOVInference.py",
        "--config",
        str(Path(pp_dir, "pretrained", "pointpillars_ov_config.json")),
        "--pc_path",
        str(Path(pp_dir, "pointpillars", "dataset", "demo_data", "test", "000002.bin")),
    ],
    cwd=str(pp_dir),
)

In [ ]:
# Run test with OpenVINO for Kitti evaluation
run(
    [
        sys.executable,
        "test-e2eOV.py",
        "--config",
        str(Path(pp_dir, "pretrained", "pointpillars_ov_config.json")),
        "--pc_path",
        str(Path(pp_dir, "pointpillars", "dataset", "demo_data", "test", "000002.bin")),
        "--device",
        device.value,
    ],
    cwd=str(pp_dir),
)

# With nn in GPU
# run([sys.executable, "test-e2eOV.py", "--config", str(Path(pp_dir, "pretrained", "pointpillars_ov_config.json")),
#     "--pc_path", str(Path(pp_dir, "pointpillars", "dataset", "demo_data", "test", "000002.bin")), "--device", "GPU"], cwd=str(pp_dir))

[back to top ⬆️](#Table-of-contents:)

#### **KITTI bin to PCD:**

To convert KITTI dataset files to PCD format, python scripts are provided in the `utils` folder of the PointPillars repository. 
Below are example commands to convert a single `.bin` file and an entire KITTI split to PCD format.

In [ ]:
# Convert single KITTI .bin file to .pcd
run([sys.executable, "pointpillars/utils/convert_bin_to_pcd.py", "pointpillars/dataset/demo_data/test/000002.bin", "--overwrite", "--verify"], cwd=str(pp_dir))

# Convert entire KITTI val split to .pcd files
# python pointpillars/utils/convert_kitti_split_to_pcd.py --data-root <PATH_TO_KITTI_DATASET> --split val  --overwrite --verify

[back to top ⬆️](#Table-of-contents:)